In [4]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier, HistGradientBoostingClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Load Data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# The first column is the ID, last column in train is target_class
X = train.iloc[:, 1:-1]
y = train["target_class"]
X_test = test.iloc[:, 1:]
test_ids = test.iloc[:, 0]


# 2. Advanced Feature Engineering
def engineering(df):
    df = df.copy()

    # Activity Intensity
    df["total_interactions"] = df[[col for col in df.columns if "int_cat" in col]].sum(
        axis=1
    )
    df["intensity_ratio"] = df["int_n"] / (df["ses_n"] + 1)

    # Recency Dynamics
    # Users whose last session is much longer ago than their average are likely churners
    df["recency_vs_avg"] = df["ses_rec"] / (df["ses_rec_avg"] + 1)

    # Value Features
    df["revenue_per_session"] = df["rev_sum"] / (df["ses_n"] + 1)

    # Binary flags for "Inactive" behaviors
    df["is_new_user"] = (df["user_rec"] <= df["ses_rec"]).astype(int)
    df["zero_revenue"] = (df["rev_sum"] == 0).astype(int)

    return df


X = engineering(X)
X_test = engineering(X_test)

# 3. Model Definitions
# We use three heavy-hitters with slightly different "views" of the data
lgbm_params = {
    "n_estimators": 1000,
    "learning_rate": 0.03,
    "num_leaves": 64,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "verbose": -1,
}

xgb_params = {
    "n_estimators": 1000,
    "learning_rate": 0.03,
    "max_depth": 7,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "use_label_encoder": False,
    "eval_metric": "logloss",
}


# 4. Pipeline with SMOTE
# SMOTE is applied only to training data via the imblearn Pipeline
def create_model(clf):
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "smote",
                SMOTE(random_state=42, sampling_strategy=0.6),
            ),  # Calibrated oversampling
            ("classifier", clf),
        ]
    )


model1 = create_model(LGBMClassifier(**lgbm_params))
model2 = create_model(XGBClassifier(**xgb_params))
model3 = create_model(
    HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.03, random_state=42)
)

# 5. Weighted Voting Ensemble
ensemble = VotingClassifier(
    estimators=[("lgbm", model1), ("xgb", model2), ("hist", model3)],
    voting="soft",
    weights=[1.5, 1, 1],  # Giving LightGBM more weight as it usually performs best here
)

# 6. Fit and Predict
print("Fitting Ensemble (this may take a minute)...")
ensemble.fit(X, y)

# Using soft voting probabilities to decide the threshold
probs = ensemble.predict_proba(X_test)[:, 1]

# Kaggle trick: instead of 0.5, we use the mean of the training target
# to align our prediction distribution with the training distribution.
threshold = y.mean()
final_preds = (probs > threshold).astype(int)

# 7. Save
output = pd.DataFrame({"ID": test_ids, "target": final_preds})
output.to_csv("results.csv", index=False)

print(f"Done! Distribution: {output['target'].value_counts(normalize=True).to_dict()}")

Fitting Ensemble (this may take a minute)...


/home/matajur/dev/matajur/Woolf/Tier_3/04_career_strategies/test_task_churn_pred/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [21:30:29] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Done! Distribution: {1: 0.8106666666666666, 0: 0.18933333333333333}


/home/matajur/dev/matajur/Woolf/Tier_3/04_career_strategies/test_task_churn_pred/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
